In [1]:
import json
from pathlib import Path
PROJECT_ROOT = next(
    p for p in Path.cwd().parents if p.name == "LaboroTomato"
)

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

In [3]:
def build_model(name: str, num_classes: int) -> nn.Module:
    name = name.lower()
    if name == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        m = models.mobilenet_v2(weights=weights)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    if name == "efficientnet_v2_s":
        weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        m = models.efficientnet_v2_s(weights=weights)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    raise ValueError(f"Unknown model: {name}")

In [4]:
def get_target_layer(model: nn.Module, model_name: str) -> nn.Module:
    """
    Return a reasonable last-conv layer for Grad-CAM.
    Works for MobileNetV2 now, EfficientNetV2-S later.
    """
    model_name = model_name.lower()
    if model_name == "mobilenet_v2":
        # Last conv block inside features
        # Common choice: model.features[-1] (ConvBNReLU)
        return model.features[-1]
    if model_name == "efficientnet_v2_s":
        # EfficientNetV2: features is a Sequential; choose last feature block
        return model.features[-1]
    raise ValueError(f"Unknown model: {model_name}")

In [5]:
class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(_, __, output):
            self.activations = output

        def backward_hook(_, grad_input, grad_output):
            # grad_output is a tuple; [0] is gradient wrt layer output
            self.gradients = grad_output[0]

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def __call__(self, x: torch.Tensor, class_idx: int):
        """
        Returns cam (H,W) in [0,1]
        """
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)  # [1, C]
        score = logits[:, class_idx].sum()
        score.backward(retain_graph=False)

        # activations: [1, K, h, w], gradients: [1, K, h, w]
        A = self.activations
        dA = self.gradients

        # Global-average pool gradients over spatial dims -> weights: [1, K, 1, 1]
        weights = dA.mean(dim=(2, 3), keepdim=True)

        # Weighted sum over channels
        cam = (weights * A).sum(dim=1, keepdim=False)  # [1, h, w]
        cam = torch.relu(cam)

        cam = cam[0].detach().cpu().numpy()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam

In [6]:
def overlay_cam_on_image(img_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.35):
    """
    img_rgb: HxWx3 uint8
    cam: hxw float [0,1] (will be resized)
    Returns overlay uint8
    """
    H, W, _ = img_rgb.shape
    cam_img = Image.fromarray(
        (cam * 255).astype(np.uint8)).resize((W, H), resample=Image.BILINEAR)
    cam_arr = np.array(cam_img).astype(np.float32) / 255.0

    # Use matplotlib colormap (no manual colors)
    cmap = plt.get_cmap("jet")
    heat = cmap(cam_arr)[..., :3]  # drop alpha
    heat = (heat * 255).astype(np.uint8)

    overlay = (1 - alpha) * img_rgb.astype(np.float32) + \
        alpha * heat.astype(np.float32)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    return overlay

In [7]:
def main():
    # (MobileNetV2 now; reuse later for EfficientNetV2-S)
    RUN_DIR = PROJECT_ROOT / "runs_cpu_local" / \
        "baseline_mobilenetv2" / "20251228-003415"

    ckpt_path = RUN_DIR / "model_best.pt"
    class_map_path = RUN_DIR / "class_to_idx.json"
    samples_csv = RUN_DIR / "samples.csv"

    assert ckpt_path.exists(), f"Missing: {ckpt_path}"
    assert class_map_path.exists(), f"Missing: {class_map_path}"
    assert samples_csv.exists(
    ), f"Missing: {samples_csv} (run 05a_select_samples.ipynb first)"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg = ckpt.get("config", {})
    model_name = cfg.get("model_name", "mobilenet_v2")
    image_size = int(cfg.get("image_size", 224))

    with open(class_map_path, "r", encoding="utf-8") as f:
        class_to_idx = json.load(f)

    idx_to_class = {v: k for k, v in class_to_idx.items()}
    class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
    num_classes = len(class_names)

    print("Model:", model_name)
    print("Classes:", class_names)

    # Build model + load weights
    model = build_model(model_name, num_classes=num_classes)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)
    model.eval()

    target_layer = get_target_layer(model, model_name)
    cam_engine = GradCAM(model, target_layer)

    # Preprocess (must match evaluation)
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)
    tfm = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    samples = pd.read_csv(samples_csv)

    out_root = RUN_DIR / "gradcam"
    (out_root / "correct").mkdir(parents=True, exist_ok=True)
    (out_root / "wrong").mkdir(parents=True, exist_ok=True)

    for _, row in samples.iterrows():
        path = Path(row["path"])
        subset = row["subset"]
        true_label = row["true_label"]
        pred_label = row["pred_label"]
        conf = float(row["confidence"])
        sample_id = int(row["sample_id"])

        if not path.exists():
            print(f"Missing image: {path}")
            continue

        # Load image
        pil = Image.open(path).convert("RGB")
        img_rgb = np.array(pil)

        # Model input
        x = tfm(pil).unsqueeze(0).to(device)

        # Grad-CAM target class:
        # Use predicted class to explain the model decision (standard for error analysis)
        pred_idx = class_to_idx[pred_label]
        cam = cam_engine(x, class_idx=pred_idx)

        overlay = overlay_cam_on_image(img_rgb, cam, alpha=0.35)

        # Save with informative but clean name
        # Avoid “step1e”; keep stable
        safe_true = true_label.replace(" ", "_")
        safe_pred = pred_label.replace(" ", "_")

        out_name = f"{sample_id:03d}__true-{safe_true}__pred-{safe_pred}__conf-{conf:.3f}.png"
        out_path = out_root / subset / out_name

        # Write image (matplotlib-free save to preserve exact pixels)
        Image.fromarray(overlay).save(out_path)

    print(f"Saved Grad-CAM overlays to: {out_root}")

In [8]:
if __name__ == "__main__":
    main()

Device: cpu
Model: mobilenet_v2
Classes: ['fully_ripened', 'green', 'half_ripened']
Saved Grad-CAM overlays to: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\20251228-003415\gradcam


*Explainability summary (natural background baseline).*\
***A curated subset of 40 test samples (30 correct, 10 incorrect) was selected for explainability analysis using Grad-CAM. The correct subset contained an equal number of samples per class (green, half-ripened, and fully ripened), while the incorrect subset focused exclusively on the dominant error mode observed in earlier analyses, namely half-ripened tomatoes misclassified as green with high confidence.***

***For correctly classified samples, Grad-CAM activations were consistently concentrated on the tomato surface, highlighting color gradients, texture regions, and biologically relevant areas such as the stem attachment point. These attention patterns suggest that, when successful, the model relies primarily on meaningful visual cues associated with fruit ripeness.***

***In contrast, all misclassified samples in the curated error set exhibited a common attention pattern: Grad-CAM maps frequently extended beyond the tomato body and strongly activated surrounding foliage, leaf–fruit boundaries, and contextual background regions. Notably, these incorrect predictions were assigned near-maximal confidence (≈1.0), despite attention being partially or predominantly diverted away from the fruit itself.***

***Taken together, these observations indicate that misclassification under natural background conditions is driven by systematic background-dependent attention rather than random noise. This finding aligns with the observed confusion matrix and calibration results, demonstrating that the model can be confidently wrong due to reliance on contextual cues, thereby reinforcing the central reliability concerns addressed in this work.***


In [1]:
!jupyter nbconvert --to html 05b_gradcam.ipynb

[NbConvertApp] Converting notebook 05b_gradcam.ipynb to html
[NbConvertApp] Writing 311741 bytes to 05b_gradcam.html
